# Sleepbox Compass: bulk Qwen OCR on a Colab GPU

Reads Pokémon Sleep detail screenshots with Qwen3.5-9B on a Colab GPU and writes
the answers in the same format as the local read cache
(`data/private/vlm_cache/<model>-p<prompt version>/<sha256>.json`). Copy the
output folder into the local cache and run `pokesleep-box sync`: every image that
Colab already read is a cache hit, so the local run skips inference. Icon
matching, the SP check, evaluation and rendering still run locally.

**Privacy:** this sends your screenshots to Google (the Colab runtime and Google
Drive). Only use it if you accept that; the local `scan`/`sync` never uploads.

Steps:
1. Put `inbox.zip` (the screenshots; optionally more zips listed in `INPUT_ZIPS`) in `MyDrive/sleepbox-colab/` on Google Drive.
2. Runtime: a GPU (A100 or L4 recommended; T4 is slow for a 9B model).
3. Run all cells. Results go to `MyDrive/sleepbox-colab/<cache folder>/`.
   Images already present there are skipped, so re-running only reads new ones.

In [ ]:
# --- Settings -------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/sleepbox-colab"   # input zips and output cache
INPUT_ZIPS = ["inbox.zip", "eval.zip"]  # zips of screenshots inside DRIVE_DIR (missing ones are skipped)
HF_MODEL = "Qwen/Qwen3.5-9B"        # weights used on the GPU (bf16)
# The local cache folder the results must land in. It is named after the local
# model id so `pokesleep-box sync` (default model) picks the results up.
LOCAL_MODEL = "mlx-community/Qwen3.5-9B-8bit"
REPO = "git+https://github.com/YoheiOhto/sleepbox-compass@main"  # for the exact prompt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install vLLM (fast batched inference) and the project itself, which provides
# the exact prompt and response parser used locally.
!pip -q install -U vllm
!pip -q install --no-deps {REPO}

In [ ]:
import hashlib, json, os, re, zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

from pokesleep_box.vlm import PROMPT, PROMPT_VERSION, parse_response

slug = re.sub(r"[^A-Za-z0-9._-]", "_", LOCAL_MODEL)
out_dir = Path(DRIVE_DIR) / f"{slug}-p{PROMPT_VERSION}"
out_dir.mkdir(parents=True, exist_ok=True)

work = Path("/content/inbox")
work.mkdir(exist_ok=True)
for name in INPUT_ZIPS:
    source = Path(DRIVE_DIR) / name
    if not source.exists():
        print("skip (not found):", source)
        continue
    with zipfile.ZipFile(source) as z:
        z.extractall(work)

suffixes = {".png", ".jpg", ".jpeg", ".webp"}
images = sorted(p for p in work.rglob("*") if p.suffix.lower() in suffixes
                and not p.name.startswith("._"))
todo = []
for p in images:
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    if not (out_dir / f"{digest}.json").exists():
        todo.append((p, digest))
print(f"images: {len(images)}, already read: {len(images) - len(todo)}, to read: {len(todo)}")
print("output:", out_dir)

In [ ]:
# Batched inference with vLLM. Same prompt, greedy decoding and no thinking,
# like the local reader (pokesleep_box.vlm.QwenReader).
import base64, time
from vllm import LLM, SamplingParams

llm = LLM(model=HF_MODEL, max_model_len=8192, limit_mm_per_prompt={"image": 1},
          gpu_memory_utilization=0.9)
params = SamplingParams(temperature=0.0, max_tokens=500)

def conversation(path):
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    data = base64.b64encode(path.read_bytes()).decode()
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}},
        {"type": "text", "text": PROMPT}]}]

BATCH = 32
start = time.time()
for i in range(0, len(todo), BATCH):
    chunk = todo[i:i + BATCH]
    results = llm.chat([conversation(p) for p, _ in chunk], params,
                       chat_template_kwargs={"enable_thinking": False}, use_tqdm=False)
    for (p, digest), result in zip(chunk, results):
        text = result.outputs[0].text
        (out_dir / f"{digest}.json").write_text(json.dumps({"text": text}, ensure_ascii=False))
    done = min(i + BATCH, len(todo))
    print(f"{done}/{len(todo)}  {(time.time() - start) / max(done, 1):.2f} s/image")

In [ ]:
# Sanity check: how many answers parse into a detail-screen read.
ok = other = bad = 0
for f in out_dir.glob("*.json"):
    screen = parse_response(json.loads(f.read_text())["text"]).get("screen")
    ok += screen == "detail"; other += screen == "other"; bad += screen == "unreadable"
print(f"detail: {ok}, other screens: {other}, unreadable: {bad}")

### If vLLM does not support the model yet

Skip the vLLM cell and run the one below instead (Hugging Face transformers, one
image at a time; slower but simple).

In [ ]:
import time, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(HF_MODEL)
model = AutoModelForImageTextToText.from_pretrained(HF_MODEL, torch_dtype=torch.bfloat16,
                                                    device_map="cuda")
start = time.time()
for n, (p, digest) in enumerate(todo, 1):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True,
                                           enable_thinking=False)
    inputs = processor(text=[prompt], images=[Image.open(p).convert("RGB")],
                       return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    text = processor.batch_decode(output[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]
    (out_dir / f"{digest}.json").write_text(json.dumps({"text": text}, ensure_ascii=False))
    print(f"{n}/{len(todo)}  {(time.time() - start) / n:.2f} s/image")